In [ ]:
import os
from dataclasses import dataclass
from typing import Tuple
import pandas as pd
import torch
from IPython.display import display
from lightning.pytorch.loggers import CSVLogger
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, random_split, TensorDataset
from torchmetrics.classification import BinaryAccuracy
from transformers import AutoTokenizer, AutoModel
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from tqdm import tqdm

In [ ]:
path = 'author_references_Oct3rd_balanced.h5'

In [ ]:
author_references = pd.read_hdf(path)

In [ ]:
model_data = author_references[[ 'title', 'abstract', 'title2', 'abstract2','label']]

In [ ]:
model_data["model_labels"] = model_data["label"].astype(int)

In [ ]:
model_data.reset_index(drop = True, inplace = True)

In [ ]:
author1_specter_data = model_data[['title', 'abstract']].to_dict(orient='records')
author2_specter_data = model_data[['title2', 'abstract2']].to_dict(orient='records')

In [ ]:
!nvidia-smi

In [ ]:
from transformers import AutoTokenizer, AutoModel

# load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained('allenai/specter2_base')

#load base model
model = AutoModel.from_pretrained('allenai/specter2_base')

#load the adapter(s) as per the required task, provide an identifier for the adapter in load_as argument and activate it
model.load_adapter("allenai/specter2_classification", source="hf", load_as="classification", set_active=True)

In [ ]:
!nvidia-smi

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
%%time
import torch

# Initialize a list to store the final embeddings
author1_embeddings = []
author2_embeddings = []

# Check the number of available GPUs
num_gpus = torch.cuda.device_count()

if num_gpus < 2:
    raise RuntimeError("This code requires at least two GPUs for parallel processing.")

author1_text_batch = [d['title'] + tokenizer.sep_token + d['abstract'] for d in list(author1_specter_data)]
author2_text_batch = [d['title2'] + tokenizer.sep_token + d['abstract2'] for d in list(author2_specter_data)]

# Split the batch into sub-batches for each GPU
author1_sub_batches = [author1_text_batch[i::num_gpus] for i in range(num_gpus)]
author2_sub_batches = [author2_text_batch[i::num_gpus] for i in range(num_gpus)]

def process_batch(sub_batch, model, tokenizer, device):
    # Process a sub-batch on a single GPU
    embeddings = []
    for batch in sub_batch:
            # Concatenate abstracts for the current sub-batch using 'title' and 'abstract'

        # Tokenize and process the sub-batch
        inputs = tokenizer(batch, padding=True, truncation=True,
                           return_tensors="pt", return_token_type_ids=False, max_length=512).to(device)

        # Process the sub-batch on the GPU
        with torch.no_grad():
            output = model(**inputs)

        # Take the first token in the sub-batch as the embedding for each example
        batch_embeddings = output.last_hidden_state[:, 0, :]
        embeddings.append(batch_embeddings)

    return torch.cat(embeddings, dim=0)

total_sub_batches = len(author1_sub_batches)

# Process each sub-batch on a separate GPU
for i in range(num_gpus):
    model.to(device)  # Move the model to the current GPU

    author1_sub_embeddings = process_batch(author1_sub_batches[i], model, tokenizer, device)
    author2_sub_embeddings = process_batch(author2_sub_batches[i], model, tokenizer, device)

    # Move the embeddings to a common device (e.g., cuda:0)
    author1_sub_embeddings = author1_sub_embeddings.to("cuda:0")
    author2_sub_embeddings = author2_sub_embeddings.to("cuda:1")

    author1_embeddings.append(author1_sub_embeddings)
    author2_embeddings.append(author2_sub_embeddings)
    

# Concatenate embeddings from all GPUs
author1_embeddings = torch.cat(author1_embeddings, dim=0)
author2_embeddings = torch.cat(author2_embeddings, dim=0)

In [ ]:
%%time

# Initialize a list to store the final embeddings
author1_embeddings = []
author2_embeddings = []

# Check the number of available GPUs
num_gpus = torch.cuda.device_count()

if num_gpus < 2:
    raise RuntimeError("This code requires at least two GPUs for parallel processing.")

author1_text_batch = [d['title'] + tokenizer.sep_token + d['abstract'] for d in list(author1_specter_data)]
author2_text_batch = [d['title2'] + tokenizer.sep_token + d['abstract2'] for d in list(author2_specter_data)]

# Split the batch into two halves
half_batch_size = len(author1_text_batch) // 2
author1_text_batch_1 = author1_text_batch[:half_batch_size]
author1_text_batch_2 = author1_text_batch[half_batch_size:]
author2_text_batch_1 = author2_text_batch[:half_batch_size]
author2_text_batch_2 = author2_text_batch[half_batch_size:]

def process_batch(sub_batch, model, tokenizer, device):
    # Process a sub-batch on a single GPU
    embeddings = []
    for batch in sub_batch:
            # Concatenate abstracts for the current sub-batch using 'title' and 'abstract'

        # Tokenize and process the sub-batch
        inputs = tokenizer(batch, padding=True, truncation=True,
                           return_tensors="pt", return_token_type_ids=False, max_length=512).to(device)

        # Process the sub-batch on the GPU
        with torch.no_grad():
            output = model(**inputs)

        # Take the first token in the sub-batch as the embedding for each example
        batch_embeddings = output.last_hidden_state[:, 0, :]
        embeddings.append(batch_embeddings)

    return torch.cat(embeddings, dim=0)

# Define a function to process a batch
def process_and_save(author_text_batch, author_embeddings, file_name):
    num_gpus = torch.cuda.device_count()

    # Split the batch into sub-batches for each GPU
    author_sub_batches = [author_text_batch[i::num_gpus] for i in range(num_gpus)]

    # Process each sub-batch on a separate GPU
    for i in range(num_gpus):
        model.to(device)  # Move the model to the current GPU

        author_sub_embeddings = process_batch(author_sub_batches[i], model, tokenizer, device)

        # Move the embeddings to a common device (e.g., cuda:0)
        author_sub_embeddings = author_sub_embeddings.to("cuda:0")

        author_embeddings.append(author_sub_embeddings)

    # Concatenate embeddings from all GPUs
    author_embeddings = torch.cat(author_embeddings, dim=0)

    # Save the embeddings to a file using an f-string
    torch.save(author_embeddings, f'{file_name}_specter.pth')

# Process and save the first half
process_and_save(author1_text_batch_1, author1_embeddings, 'author1_embeddings_1')
process_and_save(author2_text_batch_1, author2_embeddings, 'author2_embeddings_1')

# Clear CUDA memory
torch.cuda.empty_cache()

# Process and save the second half
process_and_save(author1_text_batch_2, author1_embeddings, 'author1_embeddings_2')
process_and_save(author2_text_batch_2, author2_embeddings, 'author2_embeddings_2')


In [ ]:
!nvidia-smi

In [ ]:
# Use torch.save to save the embeddings tensor to the specified file
torch.save(author1_embeddings, 'author1_embeddings_specter.pth')
torch.save(author2_embeddings, 'author2_embeddings_specter.pth')

In [ ]:
# To load the embeddings back later, you can use torch.load
loaded_embeddings = torch.load('author1_embeddings_specter')